In [1]:
import pandas as pd
import requests
import json
import os
import time

folder = r"C:\Users\LENOVO\OneDrive - London Metropolitan University\Documents\hexagon-research"

file_path = os.path.join(
    folder,
    "Hexagon-Dataset - Constitution&Law (1).csv"
)

df = pd.read_csv(file_path)

print("File:", os.path.basename(file_path))
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 2 rows:")
display(df.head(2))

File: Hexagon-Dataset - Constitution&Law (1).csv
Rows: 917
Columns: 18

Columns:
['Question ID', 'Domain', 'Subdomain', 'Question Type', 'Difficulty', 'English Question', 'Nepali Question', 'Gold Answer (EN)', 'Gold Answer (NE)', 'Explanation', 'Source ID', 'Source Title', 'Source URL', 'Source Type', 'Done', 'Reviewed By', 'Review Status', 'Notes']

First 2 rows:


,Question ID,Domain,Subdomain,Question Type,Difficulty,English Question,Nepali Question,Gold Answer (EN),Gold Answer (NE),Explanation,Source ID,Source Title,Source URL,Source Type,Done,Reviewed By,Review Status,Notes
0,CL001,Constitution & Law,Preliminary,Fact Retrieval,Easy,What is the fundamental law of Nepal?,नेपालको मौलिक कानून के हो?,The Constitution of Nepal,नेपालको संविधान,Article 1 declares the Constitution as the fun...,Art. 1,Constitution of Nepal (2015),https://lawcommission.gov.np/en/?cat=152,Constitution,Aayesha,NaN,NaN,NaN
1,CL002,Constitution & Law,Preliminary,Fact Retrieval,Easy,"According to the Constitution, in whom is the ...",संविधानअनुसार नेपालको सार्वभौमसत्ता तथा राज्यस...,The Nepalese people,नेपाली जनता,Article 2 vests sovereignty and state authorit...,Art. 2,Constitution of Nepal (2015),https://lawcommission.gov.np/en/?cat=152,Constitution,Aayesha,NaN,NaN,NaN


In [2]:
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "gemma3:1b"

response = requests.get("http://localhost:11434/api/tags")

print("Status:", response.status_code)

if response.status_code == 200:
    print("Ollama connection: OK")
else:
    print("Ollama connection FAILED")

Status: 200
Ollama connection: OK


In [3]:
dataset_context = df.to_csv(index=False)

print("Dataset characters:", len(dataset_context))
print("Approximate tokens:", len(dataset_context) // 4)

Dataset characters: 481168
Approximate tokens: 120292


In [4]:
CHUNK_SIZE = 150

all_results = []

num_chunks = (len(df) + CHUNK_SIZE - 1) // CHUNK_SIZE

print("Total questions:", len(df))
print("Chunk size:", CHUNK_SIZE)
print("Number of chunks:", num_chunks)

for chunk_no, start in enumerate(range(0, len(df), CHUNK_SIZE), start=1):

    chunk = df.iloc[start:start + CHUNK_SIZE].copy()

    chunk_context = chunk.to_csv(index=False)

    print("\n" + "=" * 70)
    print(f"CHUNK {chunk_no}/{num_chunks}")
    print(f"Rows {start + 1} to {start + len(chunk)}")
    print("Context characters:", len(chunk_context))
    print("Approx. tokens:", len(chunk_context) // 4)
    print("=" * 70)

    for _, row in chunk.iterrows():

        question_id = str(row["Question ID"])

        # -------------------------
        # ENGLISH
        # -------------------------

        prompt_en = f"""
You are evaluating a question-answering benchmark.

Use ONLY the information contained in the DATASET below.

Rules:
- Answer the question directly.
- Do not invent information.
- Do not use outside information.
- Do not repeat the question.
- Give only the answer.
- If the answer cannot be determined from the dataset, respond:
NOT FOUND

DATASET:
{chunk_context}

QUESTION:
{row["English Question"]}

ANSWER:
"""

        try:
            r_en = requests.post(
                OLLAMA_URL,
                json={
                    "model": MODEL,
                    "prompt": prompt_en,
                    "stream": False,
                    "options": {
                        "temperature": 0
                    }
                },
                timeout=600
            )

            if r_en.status_code == 200:
                answer_en = r_en.json().get("response", "").strip()
            else:
                answer_en = f"REQUEST ERROR {r_en.status_code}"

        except Exception as e:
            answer_en = f"REQUEST ERROR: {e}"

        all_results.append({
            "Question ID": question_id,
            "Language": "English",
            "Question": row["English Question"],
            "LLM Answer": answer_en,
            "Dataset Answer": row["Gold Answer (EN)"],
            "Manual Verdict": "",
            "Error Type": ""
        })

        print(f"{question_id} | English | done")

        # -------------------------
        # NEPALI
        # -------------------------

        prompt_ne = f"""
तपाईं question-answering benchmark को मूल्याङ्कन गर्दै हुनुहुन्छ।

तल दिइएको DATASET मा भएको जानकारी मात्र प्रयोग गर्नुहोस्।

नियम:
- प्रश्नको सिधा उत्तर दिनुहोस्।
- जानकारी नबनाउनुहोस्।
- बाहिरी जानकारी प्रयोग नगर्नुहोस्।
- प्रश्न दोहोर्याउनुहोस्।
- केवल उत्तर दिनुहोस्।
- DATASET बाट उत्तर निर्धारण गर्न नसके:
NOT FOUND
भन्नुहोस्।

DATASET:
{chunk_context}

QUESTION:
{row["Nepali Question"]}

ANSWER:
"""

        try:
            r_ne = requests.post(
                OLLAMA_URL,
                json={
                    "model": MODEL,
                    "prompt": prompt_ne,
                    "stream": False,
                    "options": {
                        "temperature": 0
                    }
                },
                timeout=600
            )

            if r_ne.status_code == 200:
                answer_ne = r_ne.json().get("response", "").strip()
            else:
                answer_ne = f"REQUEST ERROR {r_ne.status_code}"

        except Exception as e:
            answer_ne = f"REQUEST ERROR: {e}"

        all_results.append({
            "Question ID": question_id,
            "Language": "Nepali",
            "Question": row["Nepali Question"],
            "LLM Answer": answer_ne,
            "Dataset Answer": row["Gold Answer (NE)"],
            "Manual Verdict": "",
            "Error Type": ""
        })

        print(f"{question_id} | Nepali | done")

    # Save after every chunk
    temp_file = os.path.join(
        folder,
        "ConstitutionLaw_Gemma3_1B_TEMP.csv"
    )

    pd.DataFrame(all_results).to_csv(
        temp_file,
        index=False,
        encoding="utf-8-sig"
    )

print("\n========================================")
print("GEMMA RUN COMPLETE")
print("Total responses:", len(all_results))
print("Expected responses:", len(df) * 2)
print("========================================")

Total questions: 917
Chunk size: 150
Number of chunks: 7

CHUNK 1/7
Rows 1 to 150
Context characters: 78229
Approx. tokens: 19557
CL001 | English | done
CL001 | Nepali | done
CL002 | English | done
CL002 | Nepali | done
CL003 | English | done
CL003 | Nepali | done
CL004 | English | done
CL004 | Nepali | done
CL005 | English | done
CL005 | Nepali | done
CL006 | English | done
CL006 | Nepali | done
CL007 | English | done
CL007 | Nepali | done
CL008 | English | done
CL008 | Nepali | done
CL009 | English | done
CL009 | Nepali | done
CL010 | English | done
CL010 | Nepali | done
CL011 | English | done
CL011 | Nepali | done
CL012 | English | done
CL012 | Nepali | done
CL013 | English | done
CL013 | Nepali | done
CL014 | English | done
CL014 | Nepali | done
CL015 | English | done
CL015 | Nepali | done
CL016 | English | done
CL016 | Nepali | done
CL017 | English | done
CL017 | Nepali | done
CL018 | English | done
CL018 | Nepali | done
CL019 | English | done
CL019 | Nepali | done
CL020 | English

In [6]:
results_df = pd.DataFrame(all_results)

output_file = os.path.join(
    folder,
    "ConstitutionLaw_Gemma3_1B_Results.csv"
)

results_df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:")
print(output_file)

print("\nRows:", len(results_df))
print("Expected:", len(df) * 2)

Saved:
C:\Users\LENOVO\OneDrive - London Metropolitan University\Documents\hexagon-research\ConstitutionLaw_Gemma3_1B_Results.csv

Rows: 1834
Expected: 1834
